# Notebook for the analyses of the results

**Autor**: Valentin Velev

**Last modified**: 23.02.2026

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict
from typing import Optional, Literal
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

ROOT = Path.cwd().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_handler import DataHandler
from src.configs.datasets import DATASETS, filtered_beir
from src.analysis.table_generator import TableGenerator
from src.analysis.hard_queries import HardQueries
from src.analysis.mappings import (
    FAMILY_COLORS, FAMILY_MARKERS, DATASET_TITLES_PRETTY,
    KEEP_LABELS, 
    MODEL_NAMES_PRETTY, MODEL_FAMILIES_PRETTY,
    KEEP_LABELS_PRESENTATION
)
from src.analysis.misc import assign_family, get_benchmark

SCORES_ROOT = ROOT / "outputs" / "scores"

data_handler = DataHandler(
    sources=DATASETS,
    folder="data/raw"
)

if "irds:beir/msmarco/test" in filtered_beir: filtered_beir.remove("irds:beir/msmarco/test")
if "irds:beir/webis-touche2020" in filtered_beir: filtered_beir.remove("irds:beir/webis-touche2020")

for folder in ["figures", "tables"]:
    os.makedirs(folder, exist_ok=True)

## Results tables

### Load published results

In [ ]:
# load published results
published = pd.read_excel("published.xlsx", sheet_name="published (to import in Python)").set_index("model")

# split InstructIR and FollowIR
published[["instructir_robustness", "instructir_ndcg"]] = (
    published
    .pop("instructir")
    .str.split(",", expand=True)
    .astype(float)
)

published[["followir_score", "followir_pmrr"]] = (
    published
    .pop("followir")
    .str.split(",", expand=True)
    .astype(float)
)

# add InstructIR's supplementary scores (page 7 in InstructIR paper)
published.loc["instructor", "instructir_robustness"] = 0.5678
published.loc["instructor", "instructir_ndcg"] = 0.8954

### Main table with absolute differences color highlighting

In [ ]:
table_generator = TableGenerator(scores_root=SCORES_ROOT, data_handler=data_handler)

latex_tab_abs = table_generator.create_aggregated_table(digit=3, beir_avg="macro", published=published, boot=False)

with open("tables/latex_tab_abs.tex", "w", encoding="utf-8") as f:
    f.write(latex_tab_abs)

### Main table with standard deviation color highlighting

In [ ]:
# latex_tab_boot, latex_tab_boot_std_dev = table_generator.create_aggregated_table( # takes around 70 mins
#     digit=3,
#     beir_avg="macro",
#     published=published,
#     boot=True,
#     boot_cutoffs=[2,3], 
#     gen_std_dev_table=True
# )

# # paper/thesis table
# with open("tables/latex_tab_boot.tex", "w", encoding="utf-8") as f:
#     f.write(latex_tab_boot)

# # thesis appendix table
# with open("tables/latex_tab_boot_std_dev.tex", "w", encoding="utf-8") as f:
#     f.write(latex_tab_boot_std_dev)

### Appendix tables

MS MARCO and TREC-DL

In [ ]:
latex_tab_msm = table_generator.create_msmarco_trec_table(digit=3)

with open("tables/latex_tab_msm.tex", "w", encoding="utf-8") as f:
    f.write(latex_tab_msm)

BEIR

In [ ]:
latex_tab_beir = table_generator.create_beir_table(digit=3)

with open("tables/latex_tab_beir.tex", "w", encoding="utf-8") as f:
    f.write(latex_tab_beir)

LoTTE

In [ ]:
latex_tab_lotte = table_generator.create_lotte_table(digit=3)

with open("tables/latex_tab_lotte.tex", "w", encoding="utf-8") as f:
    f.write(latex_tab_lotte)

InstructIR and FollowIR

In [ ]:
latex_tab_instruct = table_generator.create_instructir_followir_table(digit=3)

with open("tables/latex_tab_instruct.tex", "w", encoding="utf-8") as f:
    f.write(latex_tab_instruct)

## Compare to published results (raw scores)

In [ ]:
display(published)

In [ ]:
def pct_diff_calc(published_results: pd.DataFrame, beir_avg: str, pct: bool = False):
    """
    ...
    """
    df_experiments = published_results.copy()
    df_delta = published_results.copy()
    if pct: df_pct_diff = published_results.copy()
    
    for model in list(published.index):
        for dataset in list(published_results.columns):
            
            try:
                # load the experiment scores
                score = table_generator._average_score(model, dataset, 3, beir_avg)
                
                # load the published scores
                score_published = published_results.loc[model, dataset]
                
                # calculate delta
                delta = float(score) - float(score_published)
                
                if pct:
                    pct_diff = delta / float(score_published) # for GritLM FollowIR p-MRR it yields error because published score is 0
                
                # overwrite original df
                df_experiments.loc[model, dataset] = round(score, 4)
                df_delta.loc[model, dataset] = round(delta, 4)
                if pct: df_pct_diff.loc[model, dataset] = round(pct_diff, 4)
                
            except FileNotFoundError:
                print(f"File not found for {model} and {dataset}")
                df_experiments.loc[model, dataset] = pd.NA
                df_delta.loc[model, dataset] = pd.NA
                if pct: df_pct_diff.loc[model, dataset] = pd.NA
                pass
    
    if pct: return df_experiments, df_delta, df_pct_diff
    else: return df_experiments, df_delta

### Which averaging for BEIR?

In [ ]:
df_exp_macro, df_diff_macro = pct_diff_calc(published, "macro")
df_exp_micro, df_diff_micro = pct_diff_calc(published, "micro")
display(df_diff_macro)
display(df_diff_micro)

### Avg performance across model families

In [ ]:
df_exp_macro["family"] = df_exp_macro.index.to_series().apply(assign_family)
df_exp_macro.groupby("family").mean(numeric_only=True)

### Avg differences across datasets

In [ ]:
df_diff_macro.abs().mean(numeric_only=True)

### Top-N models with smallest differences

In [ ]:
df_diff_macro.select_dtypes(include="number").abs().mean(axis=1).sort_values().head(10)

### Top-N model-dataset pairs with highest differences

In [ ]:
N = 25

s = df_diff_macro.stack()

s_num = pd.to_numeric(s, errors="coerce").dropna()

top_pos = s_num.sort_values(ascending=False).head(N)
top_neg = s_num.sort_values(ascending=True).head(N)

top_n = pd.concat([top_pos, top_neg])
print(top_n)

### Avg differences across model families

In [ ]:
df_diff_macro["family"] = df_diff_macro.index.to_series().apply(assign_family)
df_diff_macro.select_dtypes(include="number").abs().groupby(df_exp_macro["family"]).mean()

## Online cost - Average query latency (Results Section in Paper)

In [ ]:
def _load_json(model_name: str, filename: str) -> dict:
    path = SCORES_ROOT / model_name / filename
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _get_timing(model_name: str, filename: str, variant: Optional[Literal["og", "changed"]] = None) -> dict:
    if variant != None:
        return _load_json(model_name, filename).get("runtime", {}).get(variant, {})
    else:
        return _load_json(model_name, filename).get("timing", {}) or {}


def _calc_latency(t: dict) -> float:
    n = t.get("num_queries", 0) or 0
    if n <= 0:
        return np.nan

    qe = float(t.get("query_encoding_seconds", 0.0))
    se = float(t.get("search_seconds", 0.0))
    rr = float(t.get("rerank_seconds", 0.0))
    return 1000.0 * (qe + se + rr) / n


def average_query_latency(model_name: str, dataset_name: str) -> float:
    """
    ...
    """
    # MS MARCO / TREC
    MSMARCO_CASES = {
        "msmarco-passage": "irds_msmarco-passage_dev_small.json",
        "msmarco-passage-trec-dl-2019": "irds_msmarco-passage_trec-dl-2019_judged.json",
        "msmarco-passage-trec-dl-2020": "irds_msmarco-passage_trec-dl-2020_judged.json",
    }
    if dataset_name in MSMARCO_CASES:
        file = MSMARCO_CASES[dataset_name]
        return round(_calc_latency(_get_timing(model_name, file)), 2)

    # BEIR
    if dataset_name == "beir":
        scores = []
        cqa_scores = []

        for ds in filtered_beir:
            ds_norm = ds.replace("/", "_").replace(":", "_")
            latency = _calc_latency(_get_timing(model_name, f"{ds_norm}.json"))
            if "cqadupstack" in ds_norm:
                cqa_scores.append(latency)
            else:
                scores.append(latency)

        cqa = sum(cqa_scores) / len(cqa_scores)
        beir_val = (sum(scores) + cqa) / len(scores)
        
        return round(beir_val, 2)

    # LoTTE
    if dataset_name == "lotte":
        latency_forum = _calc_latency(_get_timing(model_name, "irds_lotte_pooled_test_forum.json"))
        latency_search = _calc_latency(_get_timing(model_name, "irds_lotte_pooled_test_search.json"))

        return round((latency_forum + latency_search) / 2, 2)

    # InstructIR
    if dataset_name == "instructir":
        return round(_calc_latency(_get_timing(model_name, "hf_kaist-ai_InstructIR.json")), 2)

    # FollowIR
    if dataset_name == "followir":
        latency_core_og = _calc_latency(_get_timing(model_name, "hf_jhu-clsp_core17-instructions.json", variant="og"))
        latency_core_changed = _calc_latency(_get_timing(model_name, "hf_jhu-clsp_core17-instructions.json", variant="changed"))
        latency_core = (latency_core_og + latency_core_changed) / 2
        
        latency_news_og = _calc_latency(_get_timing(model_name, "hf_jhu-clsp_news21-instructions.json", variant="og"))
        latency_news_changed = _calc_latency(_get_timing(model_name, "hf_jhu-clsp_news21-instructions.json", variant="changed"))
        latency_news = (latency_news_og + latency_news_changed) / 2
        
        latency_robust_og = _calc_latency(_get_timing(model_name, "hf_jhu-clsp_robust04-instructions.json", variant="og"))
        latency_robust_changed = _calc_latency(_get_timing(model_name, "hf_jhu-clsp_robust04-instructions.json", variant="changed"))
        latency_robust = (latency_robust_og + latency_robust_changed) / 2

        return round((latency_core + latency_news + latency_robust) / 3, 2)

In [ ]:
def avg_query_latency_calc():
    """
    ...
    """
    
    latencies = defaultdict(dict)
    
    for model in list(published.index):
        for dataset in ["msmarco-passage", "msmarco-passage-trec-dl-2019", "msmarco-passage-trec-dl-2020", "beir", "lotte", "instructir", "followir"]:
            
            try:
                latency = average_query_latency(model, dataset)
                latencies[model][dataset] = latency
                
            except Exception:
                print(f"File not found for {model} and {dataset}")
                latencies[model][dataset] = np.nan
                pass
    
    return {m: dict(d) for m, d in latencies.items()}

In [ ]:
df_latency = pd.DataFrame(avg_query_latency_calc())
display(df_latency)
display(df_latency.T)

### By model

In [ ]:
exclude = ["msmarco-passage-trec-dl-2019", "msmarco-passage-trec-dl-2020"]
df_latency.loc[~df_latency.index.isin(exclude)].mean(numeric_only=True)

### By family

In [ ]:
df_latency_pivot = df_latency.T
df_latency_pivot["family"] = df_latency_pivot.index.to_series().apply(assign_family)
result = (
    df_latency_pivot
    .loc[:, ~df_latency_pivot.columns.isin(exclude)]
    .groupby("family")
    .mean(numeric_only=True)
)
print(result.mean(axis=1))
display(result)

### LLM-backbone vs rest

In [ ]:
lst_llm_backbone = ["llm2vec", "repllama", "gritlm", "nvembed"]
llm_backbone = [c for c in df_latency.columns if c in lst_llm_backbone]
rest = [c for c in df_latency.columns if c not in lst_llm_backbone]

df_latency_no_trec = df_latency.copy()
df_latency_no_trec = df_latency.loc[~df_latency.index.isin(exclude)]

print("LLM backbone:", df_latency_no_trec[llm_backbone].values.mean())
print("Rest:", df_latency_no_trec[rest].values.mean())

### Runtime vs retrieval quality plot - single-column (Figure 1 in paper)

In [ ]:
df_exp_macro, df_diff_macro = pct_diff_calc(published, "macro")

# retrieval quality
quality_long = df_exp_macro.reset_index().melt(
    id_vars="model",
    var_name="dataset",
    value_name="quality"
)

QUALITY_TO_LAT_DS = {
    "instructir_ndcg": "instructir",
    "instructir_robustness": "instructir",
    "followir_score": "followir",
    "followir_pmrr": "followir",
}

quality_long["latency_dataset"] = quality_long["dataset"].replace(QUALITY_TO_LAT_DS)

# latency
latency_long = (
    df_latency
    .reset_index()
    .rename(columns={"index": "dataset"})
    .melt(
        id_vars="dataset",
        var_name="model",
        value_name="latency_ms",
    )
)

combined = quality_long.merge(
    latency_long.rename(columns={"dataset": "latency_dataset"}),
    on=["model", "latency_dataset"],
    how="left"
).drop("latency_dataset", axis=1)

combined["family"] = combined["model"].apply(assign_family)

In [ ]:
def create_runtime_plot(
    ds: str,
    df: pd.DataFrame,
    ax: plt.Axes,
    last: bool,
    title_size: int = 16,
    xlab_size: int = 16,
    ylab_size: int = 11,
    labs_pos = KEEP_LABELS,
    ylab = None,
    point_lab_size = 9
):
    d = df[df["dataset"] == ds].copy()

    for _, row in d.iterrows():
        fam = row["family"]
        color = FAMILY_COLORS.get(fam, "gray")
        marker = FAMILY_MARKERS.get(fam, "x")

        ax.scatter(
            row["latency_ms"], row["quality"],
            s=120, c=color, marker=marker,
            edgecolors="black", linewidths=0.7, alpha=0.75,
        )

        pretty = MODEL_NAMES_PRETTY.get(row["model"], row["model"])
        specs = labs_pos.get(ds, {})
        
        if pretty in specs:
            cfg = specs[pretty]

            ax.annotate(
                pretty,
                (row["latency_ms"], row["quality"]),
                xytext=(cfg.get("dx", 6), cfg.get("dy", 6)),
                textcoords="offset points",
                fontsize=point_lab_size,
                ha=cfg.get("ha", "center"),
                va=cfg.get("va", "center"),
                clip_on=True,
            )

    if last:
        ax.set_xlabel("Average Query Latency (ms/query)", fontsize=xlab_size, fontweight="bold")

    if ds != "followir_score":
        ax.set_ylim(0, 1)
    else:
        ax.set_ylim(0, 100)
        
    if ylab != None:
        ylabel = ylab
    else:
        if ds in {"msmarco-passage-trec-dl-2019", "msmarco-passage-trec-dl-2020", "beir"}:
            ylabel = "Average nDCG@10"
        elif ds == "msmarco-passage":
            ylabel = "Average MRR@10"
        elif ds == "lotte":
            ylabel = "Average Success@5"
        elif ds == "instructir_robustness":
            ylabel = r"Average" "\n" r"Robustness@10"
        else:
            ylabel = r"Average Score" "\n" r"(MAP@1000 & nDCG@5)"

    ax.set_ylabel(ylabel, fontsize=ylab_size, fontweight="bold")
    ax.set_title(DATASET_TITLES_PRETTY.get(ds, ds), fontsize=title_size, fontweight="bold")
    
    ax.tick_params(axis="both", which="major", labelsize=12)

datasets = [d for d in combined.dataset.unique() if d not in {"instructir_ndcg", "followir_pmrr"}]

n = len(datasets)
fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(13, 2.5*n), sharex=False)

if n == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    last = False
    if ds == "followir_score": last = True
    create_runtime_plot(ds, combined, ax, last=last, point_lab_size=8.5)

# one legend for the whole figure
handles = []
for fam, color in FAMILY_COLORS.items():
    if fam.lower() == "other":
        continue
    handles.append(plt.Line2D([0], [0], marker=FAMILY_MARKERS[fam], markersize=12, linestyle="", markerfacecolor=color, markeredgecolor="black", label=fam))

legend = fig.legend(
    handles=handles,
    title="Model Family",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.04),
    ncol=4,
    frameon=False,
    fontsize=12
)
legend.get_title().set_fontsize(16)
legend.get_title().set_fontweight("bold")

fig.tight_layout(rect=[0, 0, 1, 1])
fig.subplots_adjust(hspace=0.35)
fig.savefig("figures/runtime-performance-stacked-onecol.pdf", bbox_inches="tight")

plt.show()
plt.close(fig)

## Offline cost - Document encoding and index building (Figure 3 in thesis)

In [ ]:
def _get_encoding_time(model_name: str, filename: str, variant: Optional[Literal["og", "changed"]] = None) -> dict:
    if variant != None:
        document_encoding = _load_json(model_name, filename).get("runtime", {}).get(variant, {}).get("doc_encoding_seconds")
        index_building = _load_json(model_name, filename).get("runtime", {}).get(variant, {}).get("index_build_seconds")
        return document_encoding + index_building
    else:
        document_encoding = _load_json(model_name, filename).get("timing", {}).get("doc_encoding_seconds")
        index_building = _load_json(model_name, filename).get("timing", {}).get("index_build_seconds")
        return document_encoding + index_building

def offline_cost(model_name: str, dataset_name: str) -> float:
    """
    ...
    """
    # MS MARCO / TREC
    MSMARCO_CASES = {
        "msmarco-passage": "irds_msmarco-passage_dev_small.json",
        "msmarco-passage-trec-dl-2019": "irds_msmarco-passage_trec-dl-2019_judged.json",
        "msmarco-passage-trec-dl-2020": "irds_msmarco-passage_trec-dl-2020_judged.json",
    }
    if dataset_name in MSMARCO_CASES:
        file = MSMARCO_CASES[dataset_name]
        return round(_get_encoding_time(model_name, file), 2)

    # BEIR
    if dataset_name == "beir":
        scores = []
        cqa_scores = []

        for ds in filtered_beir:
            ds_norm = ds.replace("/", "_").replace(":", "_")
            latency = _get_encoding_time(model_name, f"{ds_norm}.json")
            if "cqadupstack" in ds_norm:
                cqa_scores.append(latency)
            else:
                scores.append(latency)

        cqa = sum(cqa_scores) / len(cqa_scores)
        beir_val = (sum(scores) + cqa) / len(scores)
        
        return round(beir_val, 2)

    # LoTTE
    if dataset_name == "lotte":
        latency_forum = _get_encoding_time(model_name, "irds_lotte_pooled_test_forum.json")
        latency_search = _get_encoding_time(model_name, "irds_lotte_pooled_test_search.json")

        return round((latency_forum + latency_search) / 2, 2)

    # InstructIR
    if dataset_name == "instructir":
        return round(_get_encoding_time(model_name, "hf_kaist-ai_InstructIR.json"), 2)

    # FollowIR
    if dataset_name == "followir":
        latency_core_og = _get_encoding_time(model_name, "hf_jhu-clsp_core17-instructions.json", variant="og")
        latency_core_changed = _get_encoding_time(model_name, "hf_jhu-clsp_core17-instructions.json", variant="changed")
        latency_core = (latency_core_og + latency_core_changed) / 2
        
        latency_news_og = _get_encoding_time(model_name, "hf_jhu-clsp_news21-instructions.json", variant="og")
        latency_news_changed = _get_encoding_time(model_name, "hf_jhu-clsp_news21-instructions.json", variant="changed")
        latency_news = (latency_news_og + latency_news_changed) / 2
        
        latency_robust_og = _get_encoding_time(model_name, "hf_jhu-clsp_robust04-instructions.json", variant="og")
        latency_robust_changed = _get_encoding_time(model_name, "hf_jhu-clsp_robust04-instructions.json", variant="changed")
        latency_robust = (latency_robust_og + latency_robust_changed) / 2

        return round((latency_core + latency_news + latency_robust) / 3, 2)

In [ ]:
def offline_cost_calc():
    """
    ...
    """
    
    latencies = defaultdict(dict)
    
    for model in list(published.index):
        for dataset in ["msmarco-passage", "msmarco-passage-trec-dl-2019", "msmarco-passage-trec-dl-2020", "beir", "lotte", "instructir", "followir"]:
            
            try:
                latency = offline_cost(model, dataset)
                latencies[model][dataset] = latency
                
            except Exception:
                print(f"File not found for {model} and {dataset}")
                latencies[model][dataset] = np.nan
                pass
    
    return {m: dict(d) for m, d in latencies.items()}

In [ ]:
df_offline_cost = pd.DataFrame(offline_cost_calc())
display(df_offline_cost)

In [ ]:
# correcting zero offline cost entries
## MS MARCO and TREC-DL
df_offline_cost.loc[["msmarco-passage-trec-dl-2019", "msmarco-passage-trec-dl-2020"], :] = df_offline_cost.loc["msmarco-passage"].values

## HyDE
df_offline_cost["hyde"] = df_offline_cost["contriever"].to_numpy()

display(df_offline_cost)

In [ ]:
# -------------------------
# Helpers
# -------------------------
DATASET_TITLES_PRETTY = {
    "msmarco-passage": "MS MARCO",
    "msmarco-passage-trec-dl-2019": "TREC-DL 2019",
    "msmarco-passage-trec-dl-2020": "TREC-DL 2020",
    "beir": "BEIR",
    "lotte": "LoTTE",
    "instructir": "InstructIR",
    "followir": "FollowIR",
}


def _build_model_to_family(model_families: dict[str, list[str]]) -> dict[str, str]:
    out = {}
    for fam, models in model_families.items():
        for m in models:
            out[m] = fam
    return out

MODEL_TO_FAMILY = _build_model_to_family(MODEL_FAMILIES_PRETTY)

# Desired model order: families in dict order, then models listed within each family
ORDERED_MODELS = [m for fam in MODEL_FAMILIES_PRETTY for m in MODEL_FAMILIES_PRETTY[fam]]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def barh_small_multiples_pretty_centered(
    df: pd.DataFrame,
    *,
    bar_height_per_item: float = 0.35,
    base_width: float = 6.8,
    logx: bool = False,
    show_family_legend: bool = True,
    wspace: float = 0.55,
):
    # ---- numeric ----
    plot_df = df.apply(pd.to_numeric, errors="coerce")

    # ---- order columns by family listing (then unknown at end) ----
    ordered = [m for fam in MODEL_FAMILIES_PRETTY for m in MODEL_FAMILIES_PRETTY[fam] if m in plot_df.columns]
    unknown = [m for m in plot_df.columns if m not in ordered]
    plot_df = plot_df.reindex(columns=ordered + unknown)

    # ---- pretty names ----
    plot_df = plot_df.rename(index=DATASET_TITLES_PRETTY, columns=MODEL_NAMES_PRETTY)

    # pretty model name -> family (for coloring)
    pretty_to_family = {}
    for fam, models in MODEL_FAMILIES_PRETTY.items():
        for m in models:
            if m in MODEL_NAMES_PRETTY:
                pretty_to_family[MODEL_NAMES_PRETTY[m]] = fam

    datasets = plot_df.index.tolist()
    assert len(datasets) == 7, "This layout assumes exactly 7 datasets"
    n_bars = plot_df.shape[1]

    # ---- figure size: scale with number of bars ----
    subplot_height = n_bars * bar_height_per_item + 1.6

    fig_height = subplot_height * 2.3
    fig_width = base_width * 3

    fig = plt.figure(figsize=(fig_width, fig_height), constrained_layout=False)

    # 3 plot rows + 1 legend row
    # Use 6 columns so each plot can span 2 columns.
    gs = fig.add_gridspec(
        nrows=4,
        ncols=6,
        height_ratios=[1, 1, 1, 0.08],
        wspace=wspace,
        hspace=0.25,
    )

    axes = []

    # -------------------------
    # Row 1: 3 plots (span 2/6 each)
    # -------------------------
    axes.append(fig.add_subplot(gs[0, 0:2]))  # 1
    axes.append(fig.add_subplot(gs[0, 2:4]))  # 2
    axes.append(fig.add_subplot(gs[0, 4:6]))  # 3

    # -------------------------
    # Row 2: 2 plots centered (shifted by 1 col)
    # -------------------------
    axes.append(fig.add_subplot(gs[1, 1:3]))  # 4
    axes.append(fig.add_subplot(gs[1, 3:5]))  # 5

    # -------------------------
    # Row 3: 2 plots centered (shifted by 1 col)
    # -------------------------
    axes.append(fig.add_subplot(gs[2, 1:3]))  # 6
    axes.append(fig.add_subplot(gs[2, 3:5]))  # 7

    # ---- plot ----
    for idx, (ax, ds) in enumerate(zip(axes, datasets)):
        s = plot_df.loc[ds].sort_values(ascending=True)

        labels = s.index.tolist()
        values = s.values.astype(float)

        colors = []
        
        for lbl in labels:
            fam = pretty_to_family.get(lbl)
            colors.append(FAMILY_COLORS.get(fam, "#7f7f7f"))

        # Create y positions
        y_pos = np.arange(len(labels))

        # Plot points grouped by family so each family gets its own marker
        for fam in FAMILY_MARKERS:
            idxs = [i for i, lbl in enumerate(labels) if pretty_to_family.get(lbl) == fam]
            if not idxs:
                continue

            ax.scatter(
                values[idxs],
                y_pos[idxs],
                c=[FAMILY_COLORS.get(fam, "#7f7f7f")] * len(idxs),
                marker=FAMILY_MARKERS[fam],
                s=60,
                zorder=3,
            )

        # Restore categorical y-axis labels
        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels)
        
        ax.margins(y=0.02) # space between first bar and top + last bar and bottom

        ax.set_title(str(ds), fontsize=24, weight="bold")
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.15, linewidth=0.8)
        ax.set_axisbelow(True)

        if logx:
            ax.set_xscale("log")
            ax.set_xlim(1, None)

        ax.relim()
        ax.autoscale(axis="x", tight=True)

        ax.tick_params(axis="y", labelsize=12)
        ax.tick_params(axis="x", labelsize=12)
        ax.margins(x=0.02)

    # ---- legend ----
    if show_family_legend:
        legend_ax = fig.add_subplot(gs[3, :])
        legend_ax.axis("off")

        handles = [
            Line2D(
                [0], [0],
                marker=FAMILY_MARKERS.get(fam, "o"),
                linestyle='None',
                markerfacecolor=FAMILY_COLORS[fam],
                markeredgecolor='none',
                markersize=14,
                label=fam,
            )
            for fam in MODEL_FAMILIES_PRETTY.keys()
        ]

        legend = legend_ax.legend(
            handles=handles,
            loc="center",
            ncol=len(handles),
            frameon=False,
            fontsize=15,
            title="Model Family"
        )

        legend.get_title().set_fontsize(20)
        legend.get_title().set_fontweight("bold")
    
    # axes[5] = 6th subplot, axes[6] = 7th subplot
    pos6 = axes[5].get_position()
    pos7 = axes[6].get_position()

    # horizontal center between the two axes
    x_center = 0.5 * (pos6.x0 + pos7.x1)

    # vertical position slightly below the plots
    y = pos6.y0 - 0.0125

    fig.text(
        x_center,
        y,
        "Offline Cost in Seconds (log scaled)",
        ha="center",
        va="top",
        fontsize=20,
        fontweight="bold",
    )

    return fig, axes


fig, axes = barh_small_multiples_pretty_centered(
    df_offline_cost,
    bar_height_per_item=0.38,
    base_width=7.0,
    logx=True,
    show_family_legend=True,
    wspace=3,
)
fig.savefig("figures/offline_cost.pdf", bbox_inches="tight")
plt.show()

## Hard queries

### Setup

In [ ]:
hard_queries = HardQueries(data_handler=data_handler)

if Path("hard_queries.parquet").exists():
    df_hq = pd.read_parquet("hard_queries.parquet", engine="pyarrow")
else:
    df_hq = hard_queries.build_df_all() # takes 40 min

    df_hq.to_parquet(
        "hard_queries.parquet",
        engine="pyarrow",
        compression="zstd",
        index=False,
    )

hard_10pct_df = hard_queries.select_bottom_percent_per_dataset_model(df_hq, p=0.1)
hard_10pct_df["benchmark"] = hard_10pct_df["dataset"].apply(get_benchmark)

### Create combined heatmap - MS MARCO and TREC-DL 2019 (Figure 2 in paper)

In [ ]:
mat_top = hard_queries.jaccard_matrix_for_dataset_from_selected(hard_10pct_df, dataset_id="MS MARCO", use_distance=False)
mat_bottom = hard_queries.jaccard_matrix_for_dataset_from_selected(hard_10pct_df, dataset_id="TREC-DL 2019", use_distance=False)

SWAPS = [
    ("docT5query", "DeepCT"),
    ("ColBERTv2", "SimCSE"),
    ("SPARTA", "query2doc"),
    ("TAS-B", "ANCE"),
    ("mE5 (large)", "NV-Embed-v2"),
    ("BGE (large v1.5)", "EmbeddingGemma"),
    ("mE5 (large)", "BGE (large v1.5)"),
    ("Contriever", "DRAGON+"),
    ("coCondenser", "COCO-DR (large)"),
    ("Contriever", "coCondenser")
]

for a,b in SWAPS:
    mat_top = hard_queries.swap_models(mat_top, a, b)
    mat_bottom = hard_queries.swap_models(mat_bottom, a, b)

fig = plt.figure(figsize=(12, 18))
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    width_ratios=[25, 1],
    hspace=0.35,
    wspace=0
)

ax_top = fig.add_subplot(gs[0, 0])
ax_bottom = fig.add_subplot(gs[1, 0])
cbar_ax = fig.add_subplot(gs[:, 1])
pos = cbar_ax.get_position()
cbar_ax.set_position([
    pos.x0 - 0.075,
    pos.y0 + 0.125,
    pos.width,
    pos.height * 0.7
])

hard_queries.plot_jaccard_heatmap_ax(
    mat_top,
    ax=ax_top,
    title="MS MARCO",
    use_distance=False,
    show_cbar=True,
    cbar_ax=cbar_ax
)

hard_queries.plot_jaccard_heatmap_ax(
    mat_bottom,
    ax=ax_bottom,
    title="TREC-DL 2019",
    use_distance=False,
    show_cbar=False
)

fig.savefig("figures/hard-queries-stacked-paper.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

### Create combined heatmap - MS MARCO, TREC-DL 2019, and TREC-DL 2020 (Figure 4 in thesis)

In [ ]:
mat_top_1 = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="MS MARCO", use_distance=False
)
mat_top_2 = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="TREC-DL 2019", use_distance=False
)
mat_bottom = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="TREC-DL 2020", use_distance=False
)

SWAPS = [
    ("docT5query", "DeepCT"),
    ("ColBERTv2", "SimCSE"),
    ("SPARTA", "query2doc"),
    ("TAS-B", "ANCE"),
    ("mE5 (large)", "NV-Embed-v2"),
    ("BGE (large v1.5)", "EmbeddingGemma"),
    ("mE5 (large)", "BGE (large v1.5)"),
    ("Contriever", "DRAGON+"),
    ("coCondenser", "COCO-DR (large)"),
    ("Contriever", "coCondenser"),
]
for a, b in SWAPS:
    mat_top_1 = hard_queries.swap_models(mat_top_1, a, b)
    mat_top_2 = hard_queries.swap_models(mat_top_2, a, b)
    mat_bottom = hard_queries.swap_models(mat_bottom, a, b)

fig = plt.figure(figsize=(15, 18))

gs = fig.add_gridspec(
    nrows=2, ncols=2,
    wspace=0.45, hspace=0
)

# Top row (two equal-sized heatmaps)
ax_top_left  = fig.add_subplot(gs[0, 0])
ax_top_right = fig.add_subplot(gs[0, 1])

# Bottom row: same width as ONE top heatmap (use left column only)
ax_bottom_center = fig.add_subplot(gs[1, 0])

hard_queries.plot_jaccard_heatmap_ax(
    mat_top_1, ax=ax_top_left,
    title="MS MARCO",
    use_distance=False, show_cbar=False,
    val_lab_size=4.25, label=False
)
hard_queries.plot_jaccard_heatmap_ax(
    mat_top_2, ax=ax_top_right,
    title="TREC-DL 2019",
    use_distance=False, show_cbar=False,
    val_lab_size=4.25, label=False
)
hard_queries.plot_jaccard_heatmap_ax(
    mat_bottom, ax=ax_bottom_center,
    title="TREC-DL 2020",
    use_distance=False, show_cbar=False,
    val_lab_size=4.25, label=False
)

# ---- Center bottom heatmap horizontally (same size as top ones) ----
fig.canvas.draw()

pos_top = ax_top_left.get_position()
pos_bottom = ax_bottom_center.get_position()

ax_bottom_center.set_position([
    0.5 - pos_top.width / 2,   # center horizontally
    pos_bottom.y0,
    pos_top.width,
    pos_top.height
])

# ---- Manual colorbar placement ----
fig.canvas.draw()

gap = 0.125
cbar_h = 0.016
cbar_w = 0.5
center_x = 0.195
bottom_y = ax_bottom_center.get_position().y0 - gap

cbar_ax = fig.add_axes([center_x, bottom_y, cbar_w, cbar_h])

mappable = ax_top_left.collections[0]
cbar = fig.colorbar(mappable, cax=cbar_ax, orientation="horizontal")

cbar.set_label("Jaccard Similarity", labelpad=8, fontsize=16, weight="bold")
cbar.ax.xaxis.set_label_position("bottom")
cbar.ax.xaxis.set_ticks_position("bottom")

fig.savefig("figures/hard-queries-stacked-thesis-main.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

### Create combined heatmap - Thesis Appendix

In [ ]:
mat_top_1 = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="BEIR", use_distance=False
)
mat_top_2 = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="LoTTE", use_distance=False
)
mat_bottom_1 = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="InstructIR", use_distance=False
)
mat_bottom_2 = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="FollowIR", use_distance=False
)

SWAPS = [
    ("docT5query", "DeepCT"),
    ("ColBERTv2", "SimCSE"),
    ("SPARTA", "query2doc"),
    ("TAS-B", "ANCE"),
    ("mE5 (large)", "NV-Embed-v2"),
    ("BGE (large v1.5)", "EmbeddingGemma"),
    ("mE5 (large)", "BGE (large v1.5)"),
    ("Contriever", "DRAGON+"),
    ("coCondenser", "COCO-DR (large)"),
    ("Contriever", "coCondenser"),
]
for a, b in SWAPS:
    mat_top_1 = hard_queries.swap_models(mat_top_1, a, b)
    mat_top_2 = hard_queries.swap_models(mat_top_2, a, b)
    mat_bottom_1 = hard_queries.swap_models(mat_bottom_1, a, b)
    mat_bottom_2 = hard_queries.swap_models(mat_bottom_2, a, b)

fig = plt.figure(figsize=(15, 18))

gs = fig.add_gridspec(
    nrows=2, ncols=2,
    wspace=0.45, hspace=0
)

# Top row (two equal-sized heatmaps)
ax_top_left  = fig.add_subplot(gs[0, 0])
ax_top_right = fig.add_subplot(gs[0, 1])

# Bottom row
ax_bottom_left = fig.add_subplot(gs[1, 0])
ax_bottom_right = fig.add_subplot(gs[1, 1])

hard_queries.plot_jaccard_heatmap_ax(
    mat_top_1, ax=ax_top_left,
    title="BEIR",
    use_distance=False, show_cbar=False,
    val_lab_size=4.25, label=False
)
hard_queries.plot_jaccard_heatmap_ax(
    mat_top_2, ax=ax_top_right,
    title="LoTTE",
    use_distance=False, show_cbar=False,
    val_lab_size=4.25, label=False
)
hard_queries.plot_jaccard_heatmap_ax(
    mat_bottom_1, ax=ax_bottom_left,
    title="InstructIR",
    use_distance=False, show_cbar=False,
    val_lab_size=4.25, label=False
)
hard_queries.plot_jaccard_heatmap_ax(
    mat_bottom_2, ax=ax_bottom_right,
    title="FollowIR",
    use_distance=False, show_cbar=False,
    val_lab_size=4.25, label=False
)

# ---- Center bottom heatmap horizontally (same size as top ones) ----
fig.canvas.draw()

pos_top = ax_top_left.get_position()
pos_bottom = ax_bottom_center.get_position()

ax_bottom_center.set_position([
    0.5 - pos_top.width / 2,   # center horizontally
    pos_bottom.y0,
    pos_top.width,
    pos_top.height
])

# ---- Manual colorbar placement ----
fig.canvas.draw()

gap = 0.125
cbar_h = 0.016
cbar_w = 0.5
center_x = 0.195
bottom_y = ax_bottom_center.get_position().y0 - gap

cbar_ax = fig.add_axes([center_x, bottom_y, cbar_w, cbar_h])

mappable = ax_top_left.collections[0]
cbar = fig.colorbar(mappable, cax=cbar_ax, orientation="horizontal")

cbar.set_label("Jaccard Similarity", labelpad=8, fontsize=16, weight="bold")
cbar.ax.xaxis.set_label_position("bottom")
cbar.ax.xaxis.set_ticks_position("bottom")

fig.savefig("figures/hard-queries-stacked-thesis-appendix.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

### Create combined heatmap - all

In [ ]:
DATASETS = [
    "MS MARCO",
    "TREC-DL 2019",
    "TREC-DL 2020",
    "BEIR",
    "LoTTE",
    "InstructIR",
    "FollowIR",
]

SWAPS = [
    ("docT5query", "DeepCT"),
    ("ColBERTv2", "SimCSE"),
    ("SPARTA", "query2doc"),
    ("TAS-B", "ANCE"),
    ("mE5 (large)", "NV-Embed-v2"),
    ("BGE (large v1.5)", "EmbeddingGemma"),
    ("mE5 (large)", "BGE (large v1.5)"),
    ("Contriever", "DRAGON+"),
    ("coCondenser", "COCO-DR (large)"),
    ("Contriever", "coCondenser"),
]

# ------------------------------------------------------------
# Build matrices (unchanged)
# ------------------------------------------------------------
mats = []
for ds in DATASETS:
    mat = hard_queries.jaccard_matrix_for_dataset_from_selected(
        hard_10pct_df,
        dataset_id=ds,
        use_distance=False
    )

    for a, b in SWAPS:
        mat = hard_queries.swap_models(mat, a, b)

    mats.append(mat)

# ------------------------------------------------------------
# Layout: equal-sized panels, centered rows, controllable vertical spacing
#         + controllable gap above the colorbar
# ------------------------------------------------------------
n = len(DATASETS)
assert n == 7, "This layout assumes exactly 7 datasets"

cell_w = 14
cell_h = 14

# ---- knobs you control ----
row_gap_01 = -0.09   # between top row and middle row
row_gap_12 = -0.09   # between middle row and bottom row
row_gap_23 = 0.07    # between bottom row and colorbar (increase for more space)

top_gap = 0.35       # horizontal gap between the 3 plots in row 0 (ratio units)
mid_gap = 0.55       # horizontal gap between the 2 centered plots (ratio units)

cbar_row_h = 0.03    # colorbar row height (smaller = thinner)
# ---------------------------

# Keep 2-plot rows same plot width as 3-plot row.
# Top row width sum: 1 + top_gap + 1 + top_gap + 1 = 3 + 2*top_gap
# 2-plot row width sum: pad + 1 + mid_gap + 1 + pad = 2 + mid_gap + 2*pad
# Enforce equality -> pad = (1 + 2*top_gap - mid_gap)/2
side_pad = (1 + 2 * top_gap - mid_gap) / 2
if side_pad < 0:
    raise ValueError(
        f"Invalid gaps: side_pad computed negative ({side_pad:.3f}). "
        f"Decrease mid_gap or increase top_gap."
    )

fig = plt.figure(
    figsize=(cell_w * 3 + 1.0, cell_h * 3.5 + 2.2),
    constrained_layout=False
)

# 7 rows: [top, spacer, mid, spacer, bot, spacer, colorbar]
gs = fig.add_gridspec(
    nrows=7,
    ncols=1,
    height_ratios=[1, row_gap_01, 1, row_gap_12, 1, row_gap_23, cbar_row_h],
    hspace=0.0,
    wspace=0.0
)

axes = []

# -------------------------
# Row 0: 3 plots with explicit gap columns (no wspace)
# -------------------------
subgs_top = gs[0, 0].subgridspec(
    1, 5,
    width_ratios=[1, top_gap, 1, top_gap, 1],
    wspace=0.0
)
axes.append(fig.add_subplot(subgs_top[0, 0]))  # MS MARCO
axes.append(fig.add_subplot(subgs_top[0, 2]))  # TREC-DL 2019
axes.append(fig.add_subplot(subgs_top[0, 4]))  # TREC-DL 2020

# -------------------------
# Row 2: 2 centered plots, equal size to top row
# -------------------------
subgs_mid = gs[2, 0].subgridspec(
    1, 5,
    width_ratios=[side_pad, 1, mid_gap, 1, side_pad],
    wspace=0.0
)
axes.append(fig.add_subplot(subgs_mid[0, 1]))  # BEIR
axes.append(fig.add_subplot(subgs_mid[0, 3]))  # LoTTE

# -------------------------
# Row 4: 2 centered plots, equal size to top row
# -------------------------
subgs_bot = gs[4, 0].subgridspec(
    1, 5,
    width_ratios=[side_pad, 1, mid_gap, 1, side_pad],
    wspace=0.0
)
axes.append(fig.add_subplot(subgs_bot[0, 1]))  # InstructIR
axes.append(fig.add_subplot(subgs_bot[0, 3]))  # FollowIR

# -------------------------
# Shared horizontal colorbar axis (bottom)
# -------------------------
cbar_subgs = gs[6, 0].subgridspec(
    1, 5,
    width_ratios=[1, 2.5, 6, 2.5, 1],
    wspace=0.0
)
cbar_ax = fig.add_subplot(cbar_subgs[0, 2])

# ------------------------------------------------------------
# Plot each dataset
# ------------------------------------------------------------
for i, (ds, mat, ax) in enumerate(zip(DATASETS, mats, axes)):
    show_cbar = (i == 0)
    hard_queries.plot_jaccard_heatmap_ax(
        mat,
        ax=ax,
        title=ds,
        use_distance=False,
        show_cbar=show_cbar,
        cbar_ax=cbar_ax if show_cbar else None,
        cbar_orientation="horizontal",
        title_size=34,
        tick_size=15
    )

cbar_ax.xaxis.set_ticks_position("bottom")
cbar_ax.xaxis.set_label_position("bottom")

fig.savefig("figures/hard-queries-stacked-all.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

## Spider chart of best model within each family

In [ ]:
# all_models = [m for group in MODEL_FAMILIES_PRETTY.values() for m in group]
# datasets = ["msmarco-passage", "msmarco-passage-trec-dl-2019", "msmarco-passage-trec-dl-2020", "beir", "lotte", "instructir_robustness", "followir_score"]

# res = {}

# for model in ["splade", "dragon", "llm2vec", "nvembed"]:
#     res[model] = {}
#     for dataset in datasets:
#         dataset_label = table_generator._dataset_id_to_label(dataset)
#         score = table_generator._average_score(model, dataset_label, 3, "macro")
#         if dataset == "followir_score":
#             score /= 100
#         res[model][dataset_label] = score

In [ ]:
# def plot_all_models_radar(
#     res,
#     models=None,
#     datasets_order=None,
#     title="Model comparison (spider chart)",
#     rmin=0.0,
#     rmax=1.0
# ):
#     if models is None:
#         models = list(res.keys())

#     # Use consistent dataset order
#     if datasets_order is None:
#         datasets_order = sorted({ds for m in models for ds in res[m].keys()})

#     labels = datasets_order

#     angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
#     angles += angles[:1]  # close circle

#     fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

#     for model in models:
#         data = res[model]
#         values = [data.get(lbl, np.nan) for lbl in labels]
#         values += values[:1]  # close polygon

#         ax.plot(angles, values, linewidth=2, label=model)
#         ax.fill(angles, values, alpha=0.10)

#     ax.set_xticks(angles[:-1])
#     ax.set_xticklabels(labels)

#     ax.set_ylim(rmin, rmax)
#     ax.set_title(title, pad=20)
#     ax.grid(True)

#     ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.10))
#     plt.tight_layout()
#     plt.show()


# plot_all_models_radar(
#     res,
#     datasets_order=datasets,
#     rmin=0.0,
#     rmax=1.0
# )

## Save results

In [ ]:
output_path = "results.xlsx"
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    df_diff_macro.to_excel(writer, sheet_name="deltas", index=True)
    df_latency.to_excel(writer, sheet_name="query_latencies", index=True)

## Colloquium (presentation) plots

In [ ]:
os.makedirs("figures/presentation", exist_ok=True)

### Online costs

In [ ]:
KEEP_LABELS_PRESENTATION = {
    "msmarco-passage": {
        "BM25": {"dx": -12, "dy": -15},
        "docT5query": {"dx": 10, "dy": -7, "ha": "left", "va": "center"},
        "DeepCT": {"dx": 25, "dy": 0},
        "SPARTA": {"dx": 0, "dy": -15},
        "uniCOIL": {"dx": 0, "dy": 15},
        "SPLADE-v3": {"dx": 25, "dy": 15},
        "DPR": {"dx": 0, "dy": 12},
        "ColBERTv2": {"dx": 0, "dy": 10, "ha": "right"},
        "SimCSE": {"dx": 0, "dy": -15},
        "query2doc": {"dx": 0, "dy": -15},
        "LLM2Vec": {"dx": 0, "dy": -15},
        "RepLLaMA": {"dx": -6, "dy": 0, "ha": "right"},
        "NV-Embed-v2": {"dx": 35, "dy": 10},
        "KaLM-Embedding (v2.5)": {"dx": 65, "dy": -4},
        "GritLM": {"dx": -6, "dy": -10, "ha": "right"},
    },
    
    "msmarco-passage-trec-dl-2019": {
        "BM25": {"dx": 3, "dy": 13, "ha": "right"},
        "docT5query": {"dx": 0, "dy": 15},
        "DeepCT": {"dx": 8, "dy": -8, "ha": "left"},
        "SPARTA": {"dx": 0, "dy": -15},
        "SPLADE-v3": {"dx": 0, "dy": 15},
        "DPR": {"dx": 0, "dy": 15},
        "ColBERTv2": {"dx": 0, "dy": 15},
        "SimCSE": {"dx": 0, "dy": -15},
        "GritLM": {"dx": 0, "dy": 15},
        "LLM2Vec": {"dx": 0, "dy": -15},
        "NV-Embed-v2": {"dx": 0, "dy": 15},
        "RepLLaMA": {"dx": 0, "dy": 15},
        "ANCE": {"dx": 0, "dy": -15},
        "KaLM-Embedding (v2.5)": {"dx": 6, "dy": -10, "ha": "left"},
    },

    "msmarco-passage-trec-dl-2020": {
        "BM25": {"dx": 3, "dy": 11, "ha": "right"},
        "docT5query": {"dx": 3, "dy": 11},
        "DeepCT": {"dx": 2, "dy": -15, "ha": "left"},
        "SPARTA": {"dx": 0, "dy": -15},
        "SPLADE-v3": {"dx": 0, "dy": 15},
        "DPR": {"dx": 0, "dy": 15},
        "ColBERTv2": {"dx": 0, "dy": 15},
        "SimCSE": {"dx": 0, "dy": -15},
        "InstructOR (large)": {"dx": 0, "dy": 15},
        "NV-Embed-v2": {"dx": 0, "dy": 15},
        "KaLM-Embedding (v2.5)": {"dx": 30, "dy": -15},
        "query2doc": {"dx": 8, "dy": 12},
        "uniCOIL": {"dx": 0, "dy": 15},
        "ANCE": {"dx": -6, "dy": -13, "ha": "right"},
        "RepLLaMA": {"dx": -6, "dy": 6, "ha": "right"},
        "GritLM": {"dx": 6, "dy": -13, "ha": "left"},
        "LLM2Vec": {"dx": 0, "dy": -15}
    },

    "beir": {
        "SPLADE-v3": {"dx": 0, "dy": -15},
        #"ColBERTv2": {"dx": 0, "dy": 15},
        "query2doc": {"dx": 0, "dy": 15},
        "NV-Embed-v2": {"dx": 0, "dy": 15},
        "GritLM": {"dx": 0, "dy": 15},
        "RepLLaMA": {"dx": 0, "dy": 15},
        "uniCOIL": {"dx": 0, "dy": 15},
        "DeepCT": {"dx": 7, "dy": 0, "ha": "left"},
        "SPARTA": {"dx": 0, "dy": 12, "ha": "left"},
        "DPR": {"dx": 0, "dy": -15},
        "SimCSE": {"dx": 6, "dy": -10, "ha": "left"},
        "GTE (large v1.5)": {"dx": 0, "dy": 15},
        "LLM2Vec": {"dx": 0, "dy": -15},
    },
    
    "lotte": {
        "BM25": {"dx": 3, "dy": -13, "ha": "right"},
        "DeepCT": {"dx": 0, "dy": -15},
        "SimCSE": {"dx": 0, "dy": -15},
        "SPLADE-v3": {"dx": 0, "dy": 15},
        "query2doc": {"dx": 0, "dy": -15},
        "LLM2Vec": {"dx": 0, "dy": -15},
        "NV-Embed-v2": {"dx": -35, "dy": 8},
        "GritLM": {"dx": 18, "dy": 10},
        "RepLLaMA": {"dx": 0, "dy": -15},
        "KaLM-Embedding (v2.5)": {"dx": 6, "dy": -13, "ha": "left"},
        "uniCOIL": {"dx": 0, "dy": -15},
        "SPARTA": {"dx": 3, "dy": -13, "ha": "right"},
        "DPR": {"dx": 0, "dy": -15},
        "SimLM": {"dx": 15, "dy": -12},
        #"ColBERTv2": {"dx": -8, "dy": 20},
    },

    "instructir_robustness": {
        "BM25": {"dx": 0, "dy": -15},
        "SPARTA": {"dx": 0, "dy": 15},
        "uniCOIL": {"dx": 0, "dy": 15},
        "docT5query": {"dx": 0, "dy": 15},
        "DeepCT": {"dx": 0, "dy": 15},
        "SPLADE-v3": {"dx": 0, "dy": 15},
        "ColBERTv2": {"dx": 0, "dy": 15},
        "query2doc": {"dx": 0, "dy": 15},
        "GritLM": {"dx": 0, "dy": 15},
        "LLM2Vec": {"dx": 0, "dy": -15},
        "NV-Embed-v2": {"dx": 0, "dy": 15},
        "RepLLaMA": {"dx": 0, "dy": 12},
        "DPR": {"dx": 6, "dy": 10, "ha": "left"},
        "SimCSE": {"dx": 0, "dy": -15},
    },
    
    "followir_score": {
        "BM25": {"dx": 0, "dy": -15, "ha": "right"},
        "SPLADE-v3": {"dx": 0, "dy": 15},
        "uniCOIL": {"dx": 0, "dy": 15},
        "SPARTA": {"dx": 0, "dy": -15},
        "query2doc": {"dx": 0, "dy": 15},
        "DeepCT": {"dx": 0, "dy": 15},
        "ColBERTv2": {"dx": 0, "dy": 15},
        "NV-Embed-v2": {"dx": 7, "dy": 15},
        "RepLLaMA": {"dx": -6, "dy": -15, "ha": "left"},
        "LLM2Vec": {"dx": 10, "dy": -12},
        "GritLM": {"dx": 0, "dy": 15},
        "SimLM": {"dx": 0, "dy": 10, "ha": "right"},
        "docT5query": {"dx": 0, "dy": -15, "ha": "left"},
    },
}

from src.analysis.mappings import DATASET_TITLES_PRETTY # needs reload because it was overwritten earlier

for i, ds in enumerate(datasets):
    fig, ax = plt.subplots(figsize=(10, 6))

    create_runtime_plot(
        ds=ds,
        df=combined,
        ax=ax,
        last=True,
        title_size=16,
        xlab_size=13,
        ylab_size=13,
        labs_pos=KEEP_LABELS_PRESENTATION,
        ylab = "Average Robustness@10" if i == 5 else None
    )

    plt.savefig(f"figures/presentation/runtime-performance-{ds}.pdf", bbox_inches="tight")
    plt.tight_layout()
    plt.show()

### Offline costs

In [ ]:
def offline_cost_single_plot(
    df: pd.DataFrame,
    ds: str,
    *,
    base_width: float = 7.0,
    bar_height_per_item: float = 0.38,
    logx: bool = True,
    show_family_legend: bool = True,
):
    # numeric
    plot_df = df.apply(pd.to_numeric, errors="coerce")

    # order columns
    ordered = [m for fam in MODEL_FAMILIES_PRETTY for m in MODEL_FAMILIES_PRETTY[fam] if m in plot_df.columns]
    unknown = [m for m in plot_df.columns if m not in ordered]
    plot_df = plot_df.reindex(columns=ordered + unknown)

    # pretty names
    plot_df = plot_df.rename(index=DATASET_TITLES_PRETTY, columns=MODEL_NAMES_PRETTY)

    # pretty model name -> family (based on your canonical mapping)
    pretty_to_family = {}
    for fam, models in MODEL_FAMILIES_PRETTY.items():
        for m in models:
            if m in MODEL_NAMES_PRETTY:
                pretty_to_family[MODEL_NAMES_PRETTY[m]] = fam

    if ds in DATASET_TITLES_PRETTY:
        ds_key = DATASET_TITLES_PRETTY[ds]
    else:
        ds_key = ds

    # pick the row
    if ds_key not in plot_df.index:
        raise KeyError(f"Dataset '{ds}' not found. Available: {list(plot_df.index)}")

    s = plot_df.loc[ds_key].sort_values(ascending=True)
    labels = s.index.tolist()
    values = s.values.astype(float)

    # helpers: color + marker per pretty label
    def family_of_label(lbl: str) -> str:
        # prefer exact pretty-name mapping; otherwise fall back to keyword matching
        fam = pretty_to_family.get(lbl)
        if fam is not None:
            return fam
        return assign_family(lbl)  # your keyword-based function

    families = [family_of_label(lbl) for lbl in labels]
    colors = [FAMILY_COLORS.get(fam, "#7f7f7f") for fam in families]
    markers = [FAMILY_MARKERS.get(fam, FAMILY_MARKERS.get("Other", "v")) for fam in families]

    # --- split into two groups ---
    split_idx = 16

    # swap order intentionally
    labels_left   = labels[split_idx:]
    values_left   = values[split_idx:]
    colors_left   = colors[split_idx:]
    markers_left  = markers[split_idx:]
    families_left = families[split_idx:]

    labels_right   = labels[:split_idx]
    values_right   = values[:split_idx]
    colors_right   = colors[:split_idx]
    markers_right  = markers[:split_idx]
    families_right = families[:split_idx]

    # figure height based on larger column
    max_items = max(len(labels_left), len(labels_right))
    fig_h = max_items * bar_height_per_item + 1.6

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(base_width * 1.6, fig_h),
        sharex=True
    )

    def scatter_by_marker(ax, x, y, cols, marks):
        """
        Matplotlib scatter can't take a list of markers in one call.
        So we draw one scatter per marker shape.
        """
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        cols = np.asarray(cols)
        marks = np.asarray(marks)

        for m in np.unique(marks):
            idx = np.where(marks == m)[0]
            ax.scatter(x[idx], y[idx], c=cols[idx], marker=m, s=80, zorder=3)

    # ---- LEFT PANEL ----
    y_left = np.arange(len(labels_left))
    scatter_by_marker(ax1, values_left + 1, y_left, colors_left, markers_left)
    ax1.set_yticks(y_left)
    ax1.set_yticklabels(labels_left)
    ax1.grid(axis="x", alpha=0.25)

    # ---- RIGHT PANEL ----
    y_right = np.arange(len(labels_right))
    scatter_by_marker(ax2, values_right + 1, y_right, colors_right, markers_right)
    ax2.set_yticks(y_right)
    ax2.set_yticklabels(labels_right)
    ax2.grid(axis="x", alpha=0.25)

    if logx:
        ax1.set_xscale("log")
        ax2.set_xscale("log")

    # formatting (NO per-axis xlabel/title)
    for ax in (ax1, ax2):
        ax.tick_params(axis="y", labelsize=12)
        ax.tick_params(axis="x", labelsize=12)
        ax.margins(x=0.05)  # your padding to keep dots off the spines

    # shared title + shared x label
    fig.suptitle(ds_key, fontsize=18, weight="bold", y=0.98)
    fig.supxlabel(
        "Offline Cost in Seconds (log + 1 scaled)",
        fontsize=14,
        fontweight="bold",
        y=0
    )

    plt.tight_layout()
    return fig, (ax1, ax2)


for ds in [
    "msmarco-passage",
    "msmarco-passage-trec-dl-2019",
    "msmarco-passage-trec-dl-2020",
    "beir",
    "lotte",
    "instructir_robustness",
    "followir_score",
]:
    fig, ax = offline_cost_single_plot(
        df_offline_cost.rename(index={"instructir": "instructir_robustness", "followir": "followir_score"}),
        ds=ds,
        logx=True,
        show_family_legend=False,
    )
    fig.savefig(f"figures/presentation/offline-cost-{ds}.pdf", bbox_inches="tight")
    plt.show()

### Hard queries

In [ ]:
mat_msmarco = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="MS MARCO", use_distance=False
)
mat_trecdl2019 = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="TREC-DL 2019", use_distance=False
)
mat_trecdl2020 = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="TREC-DL 2020", use_distance=False
)
mat_beir = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="BEIR", use_distance=False
)
mat_lotte = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="LoTTE", use_distance=False
)
mat_instructir = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="InstructIR", use_distance=False
)
mat_followir = hard_queries.jaccard_matrix_for_dataset_from_selected(
    hard_10pct_df, dataset_id="FollowIR", use_distance=False
)

SWAPS = [
    ("docT5query", "DeepCT"),
    ("ColBERTv2", "SimCSE"),
    ("SPARTA", "query2doc"),
    ("TAS-B", "ANCE"),
    ("mE5 (large)", "NV-Embed-v2"),
    ("BGE (large v1.5)", "EmbeddingGemma"),
    ("mE5 (large)", "BGE (large v1.5)"),
    ("Contriever", "DRAGON+"),
    ("coCondenser", "COCO-DR (large)"),
    ("Contriever", "coCondenser"),
]

for a, b in SWAPS:
    mat_msmarco = hard_queries.swap_models(mat_msmarco, a, b)
    mat_trecdl2019 = hard_queries.swap_models(mat_trecdl2019, a, b)
    mat_trecdl2020 = hard_queries.swap_models(mat_trecdl2020, a, b)
    mat_beir = hard_queries.swap_models(mat_beir, a, b)
    mat_lotte = hard_queries.swap_models(mat_lotte, a, b)
    mat_instructir = hard_queries.swap_models(mat_instructir, a, b)
    mat_followir = hard_queries.swap_models(mat_followir, a, b)

for i, mat in enumerate([mat_msmarco, mat_trecdl2019, mat_trecdl2020, mat_beir, mat_lotte, mat_instructir, mat_followir]):
    fig, ax = plt.subplots(figsize=(10, 10))

    if i == 0:
        ds = "msmarco-passage"
        title = "MS MARCO"
    if i == 1:
        ds = "msmarco-passage-trec-dl-2019"
        title = "TREC-DL 2019"
    if i == 2:
        ds = "msmarco-passage-trec-dl-2020"
        title = "TREC-DL 2020"
    if i == 3:
        ds = "beir"
        title = "BEIR"
    if i == 4:
        ds = "lotte"
        title = "LoTTE"
    if i == 5:
        ds = "instructir"
        title = "InstructIR"
    if i == 6:
        ds = "followir"
        title = "FollowIR"

    hard_queries.plot_jaccard_heatmap_ax(
        mat_df=mat,
        ax=ax,
        title=title,
        use_distance=True,
        show_cbar=True,
        cbar_shrink=0.7,
        cbar_title_size=14,
        cbar_tick_size=10
    )

    fig.savefig(f"figures/presentation/hard-queries-{ds}.pdf", bbox_inches="tight")
    plt.show()